# Stabilizer codes for QEC

Encode → Error → Detect → Measure → Conditional Correction

## Imports

In [31]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix
import numpy as np

In [22]:
# -------------------------
# Encoding
# -------------------------
def encode(qc, data, alpha, beta):
    # Prepare |ψ⟩ on first data qubit
    qc.initialize([alpha, beta], data[0])

    # Repetition encoding
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])


def encoded_state(alpha, beta):
    data = QuantumRegister(3, 'data')
    qc = QuantumCircuit(data)

    encode(qc, data, alpha, beta)

    return Statevector(qc)

# Example state
alpha = 1 / np.sqrt(2)
beta  = 1 / np.sqrt(2)

sv = encoded_state(alpha, beta)

# Qiskit uses little-endian ordering
print("|000> component:", sv[0])
print("|111> component:", sv[7])

|000> component: (0.7071067811865475+0j)
|111> component: (0.7071067811865475+0j)


In [24]:
# -------------------------
# Error detection (syndrome extraction)
# -------------------------
def error_detection(qc, data, anc):
    qc.cx(data[0], anc[0])
    qc.cx(data[1], anc[0])

    qc.cx(data[1], anc[1])
    qc.cx(data[2], anc[1])

    qc.barrier()   # clean separation for readability


# -------------------------
# Syndrome measurement
# -------------------------
def syndrome_measurement(error_wire=None):
    data = QuantumRegister(3, 'data')
    anc  = QuantumRegister(2, 'ancilla')
    c    = ClassicalRegister(2, 'syndrome')

    qc = QuantumCircuit(data, anc, c)

    encode(qc, data, alpha, beta)

    # Optional error
    if error_wire is not None:
        qc.x(data[error_wire])

    error_detection(qc, data, anc)

    qc.measure(anc[0], c[0])
    qc.measure(anc[1], c[1])

    counts = AerSimulator().run(qc, shots=1).result().get_counts()
    bits = list(counts.keys())[0]   # c[1]c[0]

    return [int(bits[1]), int(bits[0])]  # reorder → [anc0, anc1]


# Test syndromes
print("Syndrome if error on qubit 0:", syndrome_measurement(0))  # → [1, 0]
print("Syndrome if error on qubit 1:", syndrome_measurement(1))  # → [1, 1]
print("Syndrome if error on qubit 2:", syndrome_measurement(2))  # → [0, 1]

Syndrome if error on qubit 0: [1, 0]
Syndrome if error on qubit 1: [1, 1]
Syndrome if error on qubit 2: [0, 1]


In [25]:
# -------------------------
# Circuit builder (for drawing / inspection)
# -------------------------
def build_syndrome_circuit(alpha, beta, error_wire=None):
    data = QuantumRegister(3, 'data')
    anc  = QuantumRegister(2, 'ancilla')
    c    = ClassicalRegister(2, 'syndrome')

    qc = QuantumCircuit(data, anc, c)

    encode(qc, data, alpha, beta)

    if error_wire is not None:
        qc.x(data[error_wire])

    error_detection(qc, data, anc)

    qc.measure(anc[0], c[0])
    qc.measure(anc[1], c[1])

    return qc

In [27]:
qc = build_syndrome_circuit(alpha, beta, error_wire=0)
qc.draw(fold=-1)

┌─────────────────────────────┐          ┌───┐                     ░       
    data_0: ┤ Initialize(0.70711,0.70711) ├──■────■──┤ X ├──■──────────────────░───────
            └─────────────────────────────┘┌─┴─┐  │  └───┘  │                  ░       
    data_1: ───────────────────────────────┤ X ├──┼─────────┼────■────■────────░───────
                                           └───┘┌─┴─┐       │    │    │        ░       
    data_2: ────────────────────────────────────┤ X ├───────┼────┼────┼────■───░───────
                                                └───┘     ┌─┴─┐┌─┴─┐  │    │   ░ ┌─┐   
 ancilla_0: ──────────────────────────────────────────────┤ X ├┤ X ├──┼────┼───░─┤M├───
                                                          └───┘└───┘┌─┴─┐┌─┴─┐ ░ └╥┘┌─┐
 ancilla_1: ────────────────────────────────────────────────────────┤ X ├┤ X ├─░──╫─┤M├
                                                                    └───┘└───┘ ░  ║ └╥┘
syndrome: 2/══════════════════════════════════════════════════════════════════════╩══╩═
                                                                                  0  1

In [28]:
qc = build_syndrome_circuit(alpha, beta, error_wire=1)
qc.draw(fold=-1)

┌─────────────────────────────┐                                    ░       
    data_0: ┤ Initialize(0.70711,0.70711) ├──■────■─────────■──────────────────░───────
            └─────────────────────────────┘┌─┴─┐  │  ┌───┐  │                  ░       
    data_1: ───────────────────────────────┤ X ├──┼──┤ X ├──┼────■────■────────░───────
                                           └───┘┌─┴─┐└───┘  │    │    │        ░       
    data_2: ────────────────────────────────────┤ X ├───────┼────┼────┼────■───░───────
                                                └───┘     ┌─┴─┐┌─┴─┐  │    │   ░ ┌─┐   
 ancilla_0: ──────────────────────────────────────────────┤ X ├┤ X ├──┼────┼───░─┤M├───
                                                          └───┘└───┘┌─┴─┐┌─┴─┐ ░ └╥┘┌─┐
 ancilla_1: ────────────────────────────────────────────────────────┤ X ├┤ X ├─░──╫─┤M├
                                                                    └───┘└───┘ ░  ║ └╥┘
syndrome: 2/══════════════════════════════════════════════════════════════════════╩══╩═
                                                                                  0  1

In [29]:
qc = build_syndrome_circuit(alpha, beta, error_wire=2)
qc.draw(fold=-1)

┌─────────────────────────────┐                                    ░       
    data_0: ┤ Initialize(0.70711,0.70711) ├──■────■─────────■──────────────────░───────
            └─────────────────────────────┘┌─┴─┐  │         │                  ░       
    data_1: ───────────────────────────────┤ X ├──┼─────────┼────■────■────────░───────
                                           └───┘┌─┴─┐┌───┐  │    │    │        ░       
    data_2: ────────────────────────────────────┤ X ├┤ X ├──┼────┼────┼────■───░───────
                                                └───┘└───┘┌─┴─┐┌─┴─┐  │    │   ░ ┌─┐   
 ancilla_0: ──────────────────────────────────────────────┤ X ├┤ X ├──┼────┼───░─┤M├───
                                                          └───┘└───┘┌─┴─┐┌─┴─┐ ░ └╥┘┌─┐
 ancilla_1: ────────────────────────────────────────────────────────┤ X ├┤ X ├─░──╫─┤M├
                                                                    └───┘└───┘ ░  ║ └╥┘
syndrome: 2/══════════════════════════════════════════════════════════════════════╩══╩═
                                                                                  0  1

In [30]:
qc = build_syndrome_circuit(alpha, beta, error_wire=None)
qc.draw(fold=-1)

┌─────────────────────────────┐                               ░       
    data_0: ┤ Initialize(0.70711,0.70711) ├──■────■────■──────────────────░───────
            └─────────────────────────────┘┌─┴─┐  │    │                  ░       
    data_1: ───────────────────────────────┤ X ├──┼────┼────■────■────────░───────
                                           └───┘┌─┴─┐  │    │    │        ░       
    data_2: ────────────────────────────────────┤ X ├──┼────┼────┼────■───░───────
                                                └───┘┌─┴─┐┌─┴─┐  │    │   ░ ┌─┐   
 ancilla_0: ─────────────────────────────────────────┤ X ├┤ X ├──┼────┼───░─┤M├───
                                                     └───┘└───┘┌─┴─┐┌─┴─┐ ░ └╥┘┌─┐
 ancilla_1: ───────────────────────────────────────────────────┤ X ├┤ X ├─░──╫─┤M├
                                                               └───┘└───┘ ░  ║ └╥┘
syndrome: 2/═════════════════════════════════════════════════════════════════╩══╩═
                                                                             0  1

In [32]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.quantum_info import DensityMatrix
import numpy as np


def error_correction_circuit(alpha, beta, error_wire=None):
    data = QuantumRegister(3, 'data')
    anc  = QuantumRegister(2, 'ancilla')
    c    = ClassicalRegister(2, 'syndrome')

    qc = QuantumCircuit(data, anc, c)

    # -----------------
    # Encode
    # -----------------
    qc.initialize([alpha, beta], data[0])
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])

    # -----------------
    # Error
    # -----------------
    if error_wire is not None:
        qc.x(data[error_wire])

    # -----------------
    # Syndrome extraction
    # -----------------
    qc.cx(data[0], anc[0])
    qc.cx(data[1], anc[0])

    qc.cx(data[1], anc[1])
    qc.cx(data[2], anc[1])

    qc.barrier()

    # -----------------
    # Measurement
    # -----------------
    qc.measure(anc[0], c[0])
    qc.measure(anc[1], c[1])

    qc.barrier()

    # -----------------
    # Conditional correction
    # -----------------
    # syndrome bits: c[1]c[0] (big-endian in Qiskit)

    # [1,0] → 01 → integer 1 → error on qubit 0
    qc.x(data[0]).c_if(c, 1)

    # [1,1] → 11 → integer 3 → error on qubit 1
    qc.x(data[1]).c_if(c, 3)

    # [0,1] → 10 → integer 2 → error on qubit 2
    qc.x(data[2]).c_if(c, 2)

    return qc

In [35]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import StatePreparation
import numpy as np

alpha = 1 / np.sqrt(2)
beta  = 1 / np.sqrt(2)

def encode(qc, alpha, beta):
    qc.append(StatePreparation([alpha, beta]), [0])
    qc.cx(0, 1)
    qc.cx(0, 2)

def error_detection(qc):
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(1, 4)
    qc.cx(2, 4)

sim = AerSimulator()
circuits = []

for error_wire in range(3):
    qc = QuantumCircuit(5, 2)
    encode(qc, alpha, beta)
    qc.barrier()
    qc.x(error_wire)
    qc.barrier()
    error_detection(qc)
    qc.measure(3, 0)
    qc.measure(4, 1)
    circuits.append(qc)

# Decompose StatePreparation into Aer-native gates before running
circuits = transpile(circuits, sim)

results = sim.run(circuits, shots=1).result()

for error_wire, qc in enumerate(circuits):
    bits = list(results.get_counts(qc).keys())[0]
    syndrome = [int(bits[1]), int(bits[0])]
    print(f"Syndrome if error on wire {error_wire}: {syndrome}")

Syndrome if error on wire 0: [1, 0]
Syndrome if error on wire 1: [1, 1]
Syndrome if error on wire 2: [0, 1]


In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import DensityMatrix, state_fidelity
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
# Constants
# ─────────────────────────────────────────────────────────────────────────────
alpha = 1 / np.sqrt(2)
beta  = 1 / np.sqrt(2)
sim   = AerSimulator()      # one shared instance

# Maps measured (anc0, anc1) → data qubit to correct; None = no error
SYNDROME_TABLE = {
    (0, 0): None,
    (1, 0): 0,
    (1, 1): 1,
    (0, 1): 2,
}

# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────
def encode(qc, data, alpha, beta):
    qc.initialize([alpha, beta], data[0])
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])


def error_detection(qc, data, anc):
    qc.cx(data[0], anc[0])
    qc.cx(data[1], anc[0])
    qc.cx(data[1], anc[1])
    qc.cx(data[2], anc[1])
    qc.barrier()


# ─────────────────────────────────────────────────────────────────────────────
# Encoding verification  (statevector / density matrix of ideal state)
# ─────────────────────────────────────────────────────────────────────────────
def ideal_encoded_dm(alpha, beta):
    """Density matrix of α|000⟩ + β|111⟩ — the reference for fidelity checks."""
    data = QuantumRegister(3, 'data')
    qc   = QuantumCircuit(data)
    encode(qc, data, alpha, beta)
    return DensityMatrix(qc)


# ─────────────────────────────────────────────────────────────────────────────
# Syndrome measurement  (replaces old build_syndrome_circuit + syndrome_measurement)
# ─────────────────────────────────────────────────────────────────────────────
def build_syndrome_circuit(alpha, beta, error_wire=None):
    data = QuantumRegister(3, 'data')
    anc  = QuantumRegister(2, 'ancilla')
    c    = ClassicalRegister(2, 'syndrome')
    qc   = QuantumCircuit(data, anc, c)

    encode(qc, data, alpha, beta)
    if error_wire is not None:
        qc.x(data[error_wire])
        qc.barrier()
    error_detection(qc, data, anc)
    qc.measure(anc[0], c[0])
    qc.measure(anc[1], c[1])
    return qc


def measure_syndrome(qc):
    counts = sim.run(qc, shots=1).result().get_counts()
    bits   = list(counts.keys())[0]        # Qiskit string: c[1]c[0]
    return (int(bits[1]), int(bits[0]))    # → (anc0, anc1)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Error correction
# ─────────────────────────────────────────────────────────────────────────────

# Syndrome lookup table
# ─────────────────────────────────────────────────────────────────────────────
# After measuring ancilla_0 → c[0] and ancilla_1 → c[1], Qiskit encodes the
# classical register as a single integer:
#
#   integer = c[1] × 2 + c[0] × 1   (big-endian: c[1] is the MSB)
#
#   anc0  anc1  │  c[0]  c[1]  │  integer  │  meaning
#   ────────────┼──────────────┼───────────┼──────────────────
#     0     0   │    0     0   │     0     │  no error
#     1     0   │    1     0   │     1     │  error on data[0]
#     1     1   │    1     1   │     3     │  error on data[1]
#     0     1   │    0     1   │     2     │  error on data[2]
#
# Note: the integers go 0, 1, 3, 2 — not 0, 1, 2, 3 — because the syndrome
# is not binary counting; it encodes which *parity check* failed:
#   anc0 checks (data[0], data[1]) — the left pair
#   anc1 checks (data[1], data[2]) — the right pair
#   data[1] appears in BOTH checks, so an error there lights up both ancillas.

SYNDROME_TABLE = {
    0: None,   # c = 0b00 → no error, no correction needed
    1: 0,      # c = 0b01 → anc0=1, anc1=0 → error on data[0]
    3: 1,      # c = 0b11 → anc0=1, anc1=1 → error on data[1]
    2: 2,      # c = 0b10 → anc0=0, anc1=1 → error on data[2]
}


def build_correction_circuit(alpha, beta, error_wire=None):
    """
    Full three-qubit repetition code pipeline:
        encode  →  [inject error]  →  detect  →  measure  →  correct

    Classical feedback correction uses qc.if_test, which is Qiskit's
    equivalent of PennyLane's qml.cond:

        PennyLane                         Qiskit
        ─────────────────────────────     ──────────────────────────────────
        m3 = qml.measure(3)               qc.measure(anc[0], c[0])
        m4 = qml.measure(4)               qc.measure(anc[1], c[1])
        qml.cond(m3 & ~m4, X)(wires=0)    with qc.if_test((c, 1)): qc.x(data[0])
        qml.cond(m3 &  m4, X)(wires=1)    with qc.if_test((c, 3)): qc.x(data[1])
        qml.cond(~m3 & m4, X)(wires=2)    with qc.if_test((c, 2)): qc.x(data[2])

    The if_test integers (1, 3, 2) are the classical register values from
    SYNDROME_TABLE above. Since X² = I, re-applying X on the erroneous qubit
    undoes the flip and restores the original codeword.
    """
    data = QuantumRegister(3, 'data')
    anc  = QuantumRegister(2, 'ancilla')
    c    = ClassicalRegister(2, 'syndrome')
    qc   = QuantumCircuit(data, anc, c)

    # ── Stage 1: encode ──────────────────────────────────────────────────────
    # Prepares α|000⟩ + β|111⟩ on the three data qubits.
    encode(qc, data, alpha, beta)

    # ── Stage 2: inject error ─────────────────────────────────────────────────
    # Simulates a single bit-flip on one data qubit. In a real device this
    # happens naturally; here we force it so we can test the correction logic.
    if error_wire is not None:
        qc.barrier(label=f'X error on data[{error_wire}]')
        qc.x(data[error_wire])

    # ── Stage 3: syndrome extraction ─────────────────────────────────────────
    # Four CNOT gates write parity information into the ancilla qubits:
    #
    #   ancilla_0 ^= data[0] ⊕ data[1]   ← checks the LEFT  pair
    #   ancilla_1 ^= data[1] ⊕ data[2]   ← checks the RIGHT pair
    #
    # If both data qubits in a pair agree, the ancilla stays 0.
    # If they differ (one was flipped), the ancilla flips to 1.
    # Crucially, data qubits are never measured — only the ancillas are.
    qc.barrier(label='syndrome extraction')
    error_detection(qc, data, anc)

    # ── Stage 4: measure ancillas → classical register ────────────────────────
    # ancilla_0 → c[0]  (least-significant bit)
    # ancilla_1 → c[1]  (most-significant bit)
    # Together they form the integer:  c[1]×2 + c[0]×1
    qc.measure(anc[0], c[0])
    qc.measure(anc[1], c[1])

    # ── Stage 5: classically conditioned correction ───────────────────────────
    # Each if_test checks the integer value of the classical register c.
    # Exactly one branch fires (or none if no error occurred).
    #
    #   if_test value │  syndrome bits  │  correction
    #   ──────────────┼─────────────────┼────────────────
    #        1        │  anc0=1, anc1=0 │  X on data[0]
    #        3        │  anc0=1, anc1=1 │  X on data[1]
    #        2        │  anc0=0, anc1=1 │  X on data[2]
    qc.barrier(label='correction')
    with qc.if_test((c, 1)):   # syndrome 0b01 → left-pair error → data[0]
        qc.x(data[0])
    with qc.if_test((c, 3)):   # syndrome 0b11 → both pairs    → data[1]
        qc.x(data[1])
    with qc.if_test((c, 2)):   # syndrome 0b10 → right-pair error → data[2]
        qc.x(data[2])

    return qc

In [52]:
for w in [0, 1, 2]:
    print(f"\n{'─' * 60}")
    print(f"  Full correction circuit — error on data[{w}]")
    print(f"{'─' * 60}")
    qc = build_correction_circuit(alpha, beta, error_wire=w)
    print(qc.draw(output='text', fold=-1, initial_state=True))


────────────────────────────────────────────────────────────
  Full correction circuit — error on data[0]
────────────────────────────────────────────────────────────
              ┌─────────────────────────────┐           X error on data[0] ┌───┐ syndrome extraction                      ░        correction ┌────── ┌───┐ ───────┐                                               
   data_0: |0>┤ Initialize(0.70711,0.70711) ├──■────■───────────░──────────┤ X ├──────────░────────────■──────────────────░────────────░──────┤ If-0  ┤ X ├  End-0 ├───────────────────────────────────────────────
              └─────────────────────────────┘┌─┴─┐  │           ░          └───┘          ░            │                  ░            ░      └──╥─── └───┘ ───────┘ ┌────── ┌───┐ ───────┐                        
   data_1: |0>───────────────────────────────┤ X ├──┼───────────░─────────────────────────░────────────┼────■────■────────░────────────░─────────╫───────────────────┤ If-0  ┤ X ├  End-0 ├─────────

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
# Syndrome tuple → classical register integer
# ─────────────────────────────────────────────────────────────────────────────
# measure_syndrome returns a human-readable (anc0, anc1) tuple.
# SYNDROME_TABLE is keyed by the integer Qiskit's if_test actually sees:
#   integer = c[1]×2 + c[0]×1,  where c[0]=anc0, c[1]=anc1
def syndrome_to_int(syndrome):
    anc0, anc1 = syndrome
    return anc1 * 2 + anc0


# ─────────────────────────────────────────────────────────────────────────────
# Corrected density matrix
# ─────────────────────────────────────────────────────────────────────────────
def corrected_dm(alpha, beta, error_wire=None):
    # Step 1 — measure syndrome as (anc0, anc1) tuple
    syndrome = measure_syndrome(build_syndrome_circuit(alpha, beta, error_wire))

    # Step 2 — convert to integer and look up correction qubit
    reg_int  = syndrome_to_int(syndrome)
    fix      = SYNDROME_TABLE[reg_int]

    # Step 3 — rebuild a measurement-free circuit for DensityMatrix extraction:
    #           encode → inject error → apply correction (X² = I)
    data  = QuantumRegister(3, 'data')
    qc_dm = QuantumCircuit(data)
    encode(qc_dm, data, alpha, beta)
    if error_wire is not None:
        qc_dm.x(data[error_wire])
    if fix is not None:
        qc_dm.x(data[fix])

    return DensityMatrix(qc_dm), syndrome, reg_int, fix

In [56]:
# ─────────────────────────────────────────────────────────────────────────────
# Full pipeline
# ─────────────────────────────────────────────────────────────────────────────
def run_pipeline(error_wire=None):
    label = f"data[{error_wire}]" if error_wire is not None else "none"
    print(f"\n{'─' * 52}")
    print(f"  Error injected on         : {label}")
    print(f"{'─' * 52}")

    dm_ideal                          = ideal_encoded_dm(alpha, beta)
    dm_fixed, syndrome, reg_int, fix  = corrected_dm(alpha, beta, error_wire)

    fidelity = state_fidelity(dm_ideal, dm_fixed)

    # syndrome tuple  → human-readable parity bits (anc0, anc1)
    # reg_int         → the integer if_test compares against
    # fix             → which data qubit gets the corrective X gate
    print(f"  Syndrome (anc0, anc1)     : {syndrome}")
    print(f"  Classical register int    : {reg_int}  "
          f"  (= {syndrome[1]}×2 + {syndrome[0]}×1,  if_test fires on 0x{reg_int:x})")
    print(f"  Correction applied on     : {'none' if fix is None else f'data[{fix}]'}")
    print(f"  Fidelity after correction : {fidelity:.6f}")
    print(f"  State recovered           : {'✓' if np.isclose(fidelity, 1.0) else '✗'}")

In [58]:
# ─────────────────────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # --- Encoding sanity check ---
    from qiskit.quantum_info import Statevector
    data   = QuantumRegister(3, 'data')
    qc_enc = QuantumCircuit(data)
    encode(qc_enc, data, alpha, beta)
    sv = Statevector(qc_enc)
    print("|000⟩ component:", sv[0])
    print("|111⟩ component:", sv[7])

    # --- Syndrome table verification ---
    print("\nSyndrome table:")
    for w in range(3):
        qc  = build_syndrome_circuit(alpha, beta, error_wire=w)
        syn = measure_syndrome(qc)
        print(f"  Error on data[{w}]: (anc0, anc1)={syn}  →  int={syndrome_to_int(syn)}")

    # --- Full correction pipeline ---
    for w in [None, 0, 1, 2]:
        run_pipeline(w)

    # --- Draw circuits ---
    print("\nCircuit with error on data[0] (syndrome only):")
    print(build_syndrome_circuit(alpha, beta, error_wire=0)
          .draw(output='text', fold=-1, initial_state=True))

    print("\nCircuit with error on data[1] (full correction with if_test branches):")
    print(build_correction_circuit(alpha, beta, error_wire=1)
          .draw(output='text', fold=-1, initial_state=True))

|000⟩ component: (0.7071067811865475+0j)
|111⟩ component: (0.7071067811865475+0j)

Syndrome table:
  Error on data[0]: (anc0, anc1)=(1, 0)  →  int=1
  Error on data[1]: (anc0, anc1)=(1, 1)  →  int=3
  Error on data[2]: (anc0, anc1)=(0, 1)  →  int=2

────────────────────────────────────────────────────
  Error injected on         : none
────────────────────────────────────────────────────
  Syndrome (anc0, anc1)     : (0, 0)
  Classical register int    : 0    (= 0×2 + 0×1,  if_test fires on 0x0)
  Correction applied on     : none
  Fidelity after correction : 1.000000
  State recovered           : ✓

────────────────────────────────────────────────────
  Error injected on         : data[0]
────────────────────────────────────────────────────
  Syndrome (anc0, anc1)     : (1, 0)
  Classical register int    : 1    (= 0×2 + 1×1,  if_test fires on 0x1)
  Correction applied on     : data[0]
  Fidelity after correction : 1.000000
  State recovered           : ✓

──────────────────────────────